# Setup: EvalHub Service Deployment & SDK Configuration

This notebook deploys the [EvalHub](https://github.com/eval-hub/eval-hub) service on OpenShift and configures the [eval-hub-sdk](https://github.com/eval-hub/eval-hub-sdk) to run LLM evaluations (including [lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness)) through a centralized REST API with **MLflow experiment tracking**.

## What is EvalHub?

EvalHub is a lightweight REST API service that orchestrates LLM evaluations across multiple backends. It:

- Routes evaluation requests to frameworks like **lm-evaluation-harness**, RAGAS, GuideLLM, LightEval, and more
- Tracks experiments via **MLflow** (metrics, parameters, artifacts)
- Runs natively on **OpenShift** via the TrustyAI Operator
- Supports a **"Bring Your Own Framework" (BYOF)** approach through the SDK

## EvalHub vs. LMEvalJob (1_LMEval_setup.ipynb)

| Feature | LMEvalJob (Phase 1) | EvalHub (Phase 2) |
|---------|--------------------|---------|
| Interface | Kubernetes CR (YAML) | REST API + Python SDK |
| Frameworks | lm-evaluation-harness only | Multiple (lm-eval, RAGAS, LightEval, ...) |
| Experiment tracking | Manual | Built-in MLflow integration |
| Multi-benchmark jobs | One task per CR | Multiple benchmarks per request |
| Result management | Pod logs / CR status | Centralized API + MLflow UI |

## Prerequisites

- Completed **0_model_deploy.ipynb** (model deployed on OpenShift AI)
- Completed **1_LMEval_setup.ipynb** (RBAC and secrets configured)
- TrustyAI Operator installed on the cluster

---

## Part A: Deploy EvalHub Service on OpenShift

Before using the SDK, the EvalHub service and MLflow must be running on the cluster. This section walks through the deployment.

### Step A-1: Configuration

In [1]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env", override=True)

NAMESPACE = os.getenv("NAMESPACE", "hyo-project")

print(f"Namespace: {NAMESPACE}")

Namespace: demo


### Step A-2: Verify TrustyAI Operator

The TrustyAI Operator manages the `EvalHub` Custom Resource. Verify it is installed on the cluster:

In [2]:
!oc get csv --all-namespaces 2>/dev/null | grep -i trustyai || \
    echo "TrustyAI Operator not found. Install it from OperatorHub first."

TrustyAI Operator not found. Install it from OperatorHub first.


### Step A-3: Deploy MLflow (if not already running)

EvalHub requires an MLflow tracking server to store experiment metrics and artifacts. If MLflow is not yet deployed, create one:

> **Note:** If MLflow is already running on your cluster, skip this step and note the service URL (e.g., `http://mlflow.my-namespace.svc.cluster.local:5000`).

In [3]:
mlflow_yaml = f"""apiVersion: apps/v1
kind: Deployment
metadata:
  name: mlflow
  namespace: {NAMESPACE}
spec:
  replicas: 1
  selector:
    matchLabels:
      app: mlflow
  template:
    metadata:
      labels:
        app: mlflow
    spec:
      containers:
        - name: mlflow
          image: ghcr.io/mlflow/mlflow:v2.22.0
          command: ["mlflow", "server"]
          args:
            - "--host=0.0.0.0"
            - "--port=5000"
            - "--backend-store-uri=sqlite:///mlflow/mlflow.db"
            - "--default-artifact-root=/mlflow/artifacts"
          ports:
            - containerPort: 5000
          volumeMounts:
            - name: mlflow-data
              mountPath: /mlflow
      volumes:
        - name: mlflow-data
          persistentVolumeClaim:
            claimName: mlflow-pvc
---
apiVersion: v1
kind: PersistentVolumeClaim
metadata:
  name: mlflow-pvc
  namespace: {NAMESPACE}
spec:
  accessModes: [ReadWriteOnce]
  resources:
    requests:
      storage: 5Gi
---
apiVersion: v1
kind: Service
metadata:
  name: mlflow
  namespace: {NAMESPACE}
spec:
  selector:
    app: mlflow
  ports:
    - port: 5000
      targetPort: 5000
"""

with open("/tmp/mlflow-deploy.yaml", "w") as f:
    f.write(mlflow_yaml)

print("MLflow deployment YAML generated.")
print("Review and apply with the next cell.")

MLflow deployment YAML generated.
Review and apply with the next cell.


In [4]:
!oc apply -f /tmp/mlflow-deploy.yaml
!oc rollout status deployment/mlflow -n {NAMESPACE} --timeout=120s

deployment.apps/mlflow unchanged
persistentvolumeclaim/mlflow-pvc unchanged
service/mlflow unchanged
deployment "mlflow" successfully rolled out


### Step A-4: Deploy EvalHub via the TrustyAI Operator

Create an `EvalHub` Custom Resource. The TrustyAI Operator will reconcile it into a running EvalHub service with the configured MLflow connection.

Key fields:
- `MLFLOW_TRACKING_URI` — Points to your MLflow service
- `replicas` — Number of EvalHub instances

In [5]:
MLFLOW_SVC_URL = f"http://mlflow.{NAMESPACE}.svc.cluster.local:5000"

evalhub_cr_yaml = f"""apiVersion: trustyai.opendatahub.io/v1alpha1
kind: EvalHub
metadata:
  name: evalhub
  namespace: {NAMESPACE}
spec:
  replicas: 1
  database:
    type: sqlite
  env:
    - name: MLFLOW_TRACKING_URI
      value: "{MLFLOW_SVC_URL}"
"""

with open("/tmp/evalhub-cr.yaml", "w") as f:
    f.write(evalhub_cr_yaml)

print("EvalHub CR YAML:")
print(evalhub_cr_yaml)

EvalHub CR YAML:
apiVersion: trustyai.opendatahub.io/v1alpha1
kind: EvalHub
metadata:
  name: evalhub
  namespace: demo
spec:
  replicas: 1
  database:
    type: sqlite
  env:
    - name: MLFLOW_TRACKING_URI
      value: "http://mlflow.demo.svc.cluster.local:5000"



In [6]:
!oc apply -f /tmp/evalhub-cr.yaml

evalhub.trustyai.opendatahub.io/evalhub unchanged


In [7]:
!oc describe evalhub evalhub -n {NAMESPACE}
print()
!oc get pods -n {NAMESPACE} | grep evalhub
print()
!oc logs -l app=evalhub -n {NAMESPACE} --tail=20

Name:         evalhub
Namespace:    demo
Labels:       <none>
Annotations:  kubectl.kubernetes.io/restartedAt: 2026-05-25T09:16:23Z
API Version:  trustyai.opendatahub.io/v1alpha1
Kind:         EvalHub
Metadata:
  Creation Timestamp:  2026-05-25T06:27:52Z
  Finalizers:
    trustyai.opendatahub.io/evalhub-finalizer
  Generation:        2
  Resource Version:  1294880
  UID:               3a94085b-a0eb-430f-9221-7c3f17f57cf4
Spec:
  Collections:
    leaderboard-v2
    safety-and-fairness-v1
    toxicity-and-ethical-principles
  Database:
    Max Idle Conns:  5
    Max Open Conns:  25
    Type:            sqlite
  Env:
    Name:   MLFLOW_TRACKING_URI
    Value:  http://mlflow.demo.svc.cluster.local:5000
  Providers:
    garak
    garak-kfp
    lm-evaluation-harness
  Replicas:  1
Status:
  Active Collections:
    leaderboard-v2
    safety-and-fairness-v1
    toxicity-and-ethical-principles
  Active Providers:
    garak
    garak-kfp
    lm-evaluation-harness
  Conditions:
    Last Transitio

### Step A-5: Verify EvalHub Deployment

Wait for the EvalHub pod to become ready and check its status:

In [8]:
!oc get evalhub -n {NAMESPACE}
print()
!oc get pods -n {NAMESPACE} | grep -E "evalhub|mlflow"

NAME      PHASE   READY   AGE
evalhub   Ready   True    3h41m

evalhub-7d848497bd-w6ddh                1/1     Running                  0          56m
mlflow-7969bd6d79-vvs96                 1/1     Running                  0          3h29m


### Step A-6: Get the EvalHub Service URL

The URL depends on **where this notebook is running**:

| Running From | EVALHUB_URL | Setup |
|---|---|---|
| **OpenShift Workbench** (cluster 내부) | `https://evalhub.<ns>.svc.cluster.local:8443` | 별도 설정 불필요 |
| **로컬 PC** (cluster 외부) | `https://localhost:8443` | `oc port-forward` 필요 |

In [9]:
import subprocess, socket, time, httpx

result = subprocess.run(
    ["oc", "get", "svc", "evalhub", "-n", NAMESPACE,
     "-o", "jsonpath={.metadata.name}"],
    capture_output=True, text=True,
)
svc_name = result.stdout.strip() or "evalhub"
EVALHUB_CLUSTER_URL = f"https://{svc_name}.{NAMESPACE}.svc.cluster.local:8443"
EVALHUB_LOCAL_URL = "https://localhost:8443"

def _is_port_open(port: int = 8443) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(1)
        return s.connect_ex(("127.0.0.1", port)) == 0

def _start_port_forward(namespace: str, port: int = 8443, retries: int = 6):
    """Start oc port-forward in the background if not already running."""
    if _is_port_open(port):
        print(f"Port {port} already in use — port-forward likely running.")
        return True
    print(f"Starting: oc port-forward svc/evalhub {port}:{port} -n {namespace}")
    proc = subprocess.Popen(
        ["oc", "port-forward", f"svc/evalhub", f"{port}:{port}", "-n", namespace],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    for i in range(retries):
        time.sleep(3)
        if _is_port_open(port):
            print(f"Port-forward started (pid={proc.pid}).")
            return True
        print(f"  waiting... ({(i+1)*3}s)")
    print("WARNING: port-forward did not become ready.")
    return False

_in_cluster = False
try:
    httpx.get(f"{EVALHUB_CLUSTER_URL}/api/v1/health", verify=False, timeout=3)
    _in_cluster = True
except Exception:
    pass

print("EvalHub Service URLs")
print("=" * 60)
print(f"  Cluster-internal: {EVALHUB_CLUSTER_URL}")
print(f"  Local (port-fwd): {EVALHUB_LOCAL_URL}")
print()

if _in_cluster:
    print("Detected: running INSIDE the cluster — using cluster-internal URL.")
    EVALHUB_URL = EVALHUB_CLUSTER_URL
else:
    print("Detected: running OUTSIDE the cluster — local port-forward required.")
    _start_port_forward(NAMESPACE)
    EVALHUB_URL = EVALHUB_LOCAL_URL

    try:
        resp = httpx.get(f"{EVALHUB_URL}/api/v1/health", verify=False, timeout=5)
        info = resp.json()
        print(f"\n[OK] EvalHub reachable — status={info.get('status')}, version={info.get('build')}")
    except Exception as e:
        print(f"\n[FAIL] EvalHub NOT reachable at {EVALHUB_URL}")
        print(f"       Error: {e}")
        print(f"       Run manually: oc port-forward svc/evalhub 8443:8443 -n {NAMESPACE}")

EvalHub Service URLs
  Cluster-internal: https://evalhub.demo.svc.cluster.local:8443
  Local (port-fwd): https://localhost:8443

Detected: running OUTSIDE the cluster — local port-forward required.
Port 8443 already in use — port-forward likely running.


### Step A-6b: Get the Authentication Token

EvalHub uses OpenShift's authentication system. All API calls require a **Bearer token** in the `Authorization` header. You can use:

1. **현재 로그인된 사용자의 토큰** (`oc whoami -t`) — 개발/테스트 용도. 세션 만료 시 갱신 필요
2. **ServiceAccount 토큰** — 장기 운영 용도. 별도의 SA 생성 후 토큰 발급

> **Note:** 토큰이 만료되면 `401 Unauthorized` 에러가 발생합니다. 이 경우 `oc login`으로 재로그인 후 토큰을 갱신하세요.

In [10]:
import subprocess

result = subprocess.run(["oc", "whoami", "-t"], capture_output=True, text=True)
if result.returncode == 0:
    EVALHUB_AUTH_TOKEN = result.stdout.strip()
    print(f"Token obtained: {EVALHUB_AUTH_TOKEN[:10]}...")
else:
    print("Failed to get token. Make sure you are logged in:")
    print("  oc login <cluster-url>")
    EVALHUB_AUTH_TOKEN = None

Token obtained: sha256~6bE...


### Step A-7: Health Check

Verify the EvalHub service is responding before proceeding to SDK setup:

In [31]:
import httpx

_urls = [EVALHUB_CLUSTER_URL, EVALHUB_LOCAL_URL] if _in_cluster else [EVALHUB_LOCAL_URL, EVALHUB_CLUSTER_URL]

for url in _urls:
    try:
        resp = httpx.get(f"{url}/api/v1/health", verify=False, timeout=3)
        print(f"[OK] {url}")
        print(f"     {resp.json()}")
        break
    except Exception:
        print(f"[--] {url} (unreachable)")
else:
    print("\nEvalHub is not reachable.")
    print("Attempting to (re)start port-forward...")
    if _start_port_forward(NAMESPACE):
        try:
            resp = httpx.get(f"{EVALHUB_LOCAL_URL}/api/v1/health", verify=False, timeout=3)
            print(f"\n[OK] {EVALHUB_LOCAL_URL}")
            print(f"     {resp.json()}")
        except Exception:
            print(f"[FAIL] Still unreachable after port-forward. Check 'oc' login and cluster status.")

[OK] https://localhost:8443
     {'status': 'healthy', 'timestamp': '2026-05-25T10:12:14.819580253Z', 'build': '0.3.0'}


---

## Part B: Configure the EvalHub SDK

Now that the EvalHub service is running, install and configure the Python SDK.

### Step B-1: Install the EvalHub SDK

The `eval-hub-sdk` package provides both a Python client for submitting evaluations and the adapter SDK for building custom frameworks.

In [12]:
# !pip install -q eval-hub-sdk

### Step B-2: Load Configuration

Configuration is loaded from `../.env`. Update these EvalHub-specific variables with the values discovered in Part A:

| Variable | Description | Example |
|----------|-------------|---------|
| `EVALHUB_URL` | EvalHub service endpoint (Step A-6) | `https://evalhub.my-ns.svc.cluster.local:8443` |
| `EVALHUB_AUTH_TOKEN` | **필수** — OpenShift Bearer token (Step A-6b) | `sha256~xxxx...` |
| `MLFLOW_TRACKING_URI` | MLflow server URL (Step A-3) | `http://mlflow.my-ns.svc.cluster.local:5000` |

> **Auth Token 갱신:** 토큰이 만료되면 `oc whoami -t`로 새 토큰을 발급받아 `.env`에 업데이트하세요.

In [13]:
import os
import subprocess
import httpx
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env", override=True)

NAMESPACE = os.getenv("NAMESPACE", "hyo-project")
MODEL_NAME = os.getenv("MODEL_NAME", "vllm-gemma4-e2b")
BASE_URL = os.getenv("BASE_URL", f"https://{MODEL_NAME}-predictor.{NAMESPACE}.svc.cluster.local:8443/v1")
EVALHUB_AUTH_TOKEN = os.getenv("EVALHUB_AUTH_TOKEN", None) or None
if not EVALHUB_AUTH_TOKEN:
    _r = subprocess.run(["oc", "whoami", "-t"], capture_output=True, text=True)
    if _r.returncode == 0:
        EVALHUB_AUTH_TOKEN = _r.stdout.strip()
        print("(EVALHUB_AUTH_TOKEN auto-detected from 'oc whoami -t')")
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://mlflow:5000")

EVALHUB_URL = os.getenv("EVALHUB_URL", "https://localhost:8443")
_candidates = [EVALHUB_URL, "https://localhost:8443", f"https://evalhub.{NAMESPACE}.svc.cluster.local:8443"]
for _url in dict.fromkeys(_candidates):
    try:
        httpx.get(f"{_url}/api/v1/health", verify=False, timeout=3)
        EVALHUB_URL = _url
        break
    except Exception:
        pass

print(f"Namespace:          {NAMESPACE}")
print(f"Model Name:         {MODEL_NAME}")
print(f"Model Endpoint:     {BASE_URL}")
print(f"EvalHub URL:        {EVALHUB_URL}")
print(f"Auth Token:         {'***' if EVALHUB_AUTH_TOKEN else 'None (no auth)'}")
print(f"MLflow Tracking:    {MLFLOW_TRACKING_URI}")

Namespace:          demo
Model Name:         gemma4-e2b-deployment
Model Endpoint:     https://vllm-gemma4-e2b-predictor.hyo-project.svc.cluster.local:8443/v1/completions
EvalHub URL:        https://localhost:8443
Auth Token:         ***
MLflow Tracking:    http://mlflow.demo.svc.cluster.local:5000


### Step B-3: Verify EvalHub Connectivity

Check that the SDK can connect to the EvalHub service.

In [18]:
from evalhub import SyncEvalHubClient

client = SyncEvalHubClient(
    base_url=EVALHUB_URL,
    auth_token=EVALHUB_AUTH_TOKEN,
    insecure=True,
    tenant=NAMESPACE,
)

print(f"EvalHub client initialized: {EVALHUB_URL}")
print(f"Tenant (namespace):         {NAMESPACE}")

TLS verification disabled - skipping CA bundle detection
TLS verification disabled (insecure mode)


EvalHub client initialized: https://localhost:8443
Tenant (namespace):         demo


### Step B-4: Explore Available Providers and Benchmarks

EvalHub ships with pre-configured providers. Let's list them and their benchmarks.

In [19]:
try:
    providers = client.providers.list()
    print(f"Available Providers ({len(providers)}):")
    print("=" * 60)
    for provider in providers:
        print(f"\n  Provider: {provider.name}")
        print(f"  ID:       {provider.resource.id}")
        print(f"  Desc:     {provider.description}")
        print(f"  Benchmarks: {len(provider.benchmarks)}")
except Exception as e:
    print(f"Failed to list providers: {e}")

Available Providers (4):

  Provider: LM Evaluation Harness
  ID:       lm_evaluation_harness
  Desc:     Comprehensive evaluation framework for language models with 180 benchmarks

  Benchmarks: 181

  Provider: Garak KFP
  ID:       garak-kfp
  Desc:     LLM vulnerability scanner and red-teaming framework with Data Science Pipeline execution mode.
  Benchmarks: 9

  Provider: Garak
  ID:       garak
  Desc:     LLM vulnerability scanner and red-teaming framework
  Benchmarks: 8

  Provider: Korean LM Evaluation Harness
  ID:       d949e6fc-b9a2-4402-a920-b118417ec3f5
  Desc:     Korean-language benchmarks from EleutherAI lm-evaluation-harness. Reuses the same runtime image as the built-in lm_evaluation_harness provider.

  Benchmarks: 12


In [20]:
try:
    benchmarks = client.benchmarks.list()
    print(f"\nAvailable Benchmarks ({len(benchmarks)}):")
    print("=" * 60)
    for bm in benchmarks[:20]:
        print(f"  {bm.id:30s}  category={bm.category or 'N/A':15s}  metrics={bm.metrics}")
    if len(benchmarks) > 20:
        print(f"  ... and {len(benchmarks) - 20} more")
except Exception as e:
    print(f"Failed to list benchmarks: {e}")
    benchmarks = None


Available Benchmarks (210):
  arc_easy                        category=reasoning        metrics=['acc', 'acc_norm']
  AraDiCE_boolq_lev               category=general          metrics=['acc']
  blimp                           category=general          metrics=['acc']
  blimp_anaphor_gender_agreement  category=general          metrics=['acc']
  blimp_animate_subject_trans     category=general          metrics=['acc']
  blimp_coordinate_structure_constraint_complex_left_branch  category=general          metrics=['acc']
  blimp_determiner_noun_agreement_2  category=general          metrics=['acc']
  blimp_determiner_noun_agreement_with_adj_2  category=general          metrics=['acc']
  blimp_determiner_noun_agreement_with_adjective_1  category=general          metrics=['acc']
  blimp_existential_there_object_raising  category=general          metrics=['acc']
  blimp_existential_there_subject_raising  category=general          metrics=['acc']
  blimp_intransitive              category=gen

### Register Korean Benchmarks

The default EvalHub catalog does not include Korean-language benchmarks.
We register a **custom provider** (`korean_lm_eval`) that reuses the same `lm-evaluation-harness`
runtime but exposes Korean tasks (`kmmlu`, `kobest_*`, `haerae`, `click`, etc.).

The provider definition lives in [`config/korean_lm_eval_benchmarks.yaml`](../config/korean_lm_eval_benchmarks.yaml)
following the [eval-hub-contrib](https://github.com/eval-hub/eval-hub-contrib) `provider.yaml` format.

In [21]:
import yaml, json, pathlib, httpx

KOREAN_PROVIDER_YAML = pathlib.Path("../config/korean_lm_eval_benchmarks.yaml")
KOREAN_PROVIDER_NAME = "Korean LM Evaluation Harness"

def _register_korean_provider(client, yaml_path: pathlib.Path) -> str | None:
    """Register the Korean benchmarks provider via the REST API (idempotent).
    Returns the provider_id (UUID assigned by EvalHub) or None on failure."""
    for p in client.providers.list():
        if p.name == KOREAN_PROVIDER_NAME:
            print(f"Provider '{KOREAN_PROVIDER_NAME}' already registered (id={p.resource.id}).")
            return p.resource.id

    provider_def = yaml.safe_load(yaml_path.read_text())

    resp = httpx.post(
        f"{EVALHUB_URL}/api/v1/evaluations/providers",
        headers={
            "Authorization": f"Bearer {EVALHUB_AUTH_TOKEN}",
            "Content-Type": "application/json",
            "X-Tenant": NAMESPACE,
        },
        json=provider_def,
        verify=False,
        timeout=10,
    )
    if resp.status_code in (200, 201):
        data = resp.json()
        pid = data["resource"]["id"]
        n = len(data.get("benchmarks") or [])
        print(f"Provider '{provider_def['name']}' registered (id={pid}, {n} benchmarks)")
        return pid
    else:
        print(f"Registration failed ({resp.status_code}): {resp.text[:200]}")
        return None

KOREAN_PROVIDER_ID = _register_korean_provider(client, KOREAN_PROVIDER_YAML)

benchmarks = client.benchmarks.list()
korean_keywords = ["kmmlu", "kobest", "haerae", "klue", "korean", "ko_", "click", "kbl", "kormedmcqa"]
korean_benchmarks = [
    bm for bm in benchmarks
    if any(kw in bm.id.lower() for kw in korean_keywords)
]
print(f"\nKorean Benchmarks Found: {len(korean_benchmarks)}")
print("=" * 60)
for bm in korean_benchmarks:
    print(f"  {bm.id:35s}  {bm.name}")

Provider 'Korean LM Evaluation Harness' already registered (id=d949e6fc-b9a2-4402-a920-b118417ec3f5).

Korean Benchmarks Found: 12
  kmmlu                                Korean MMLU
  kmmlu_direct                         Korean MMLU (Direct, 0-shot)
  kmmlu_direct_law                     Korean MMLU — Law (Direct)
  kobest_boolq                         KoBEST BoolQ
  kobest_copa                          KoBEST COPA
  kobest_hellaswag                     KoBEST HellaSwag
  kobest_sentineg                      KoBEST SentiNeg
  kobest_wic                           KoBEST WiC
  haerae                               HAE-RAE Bench
  click                                CLIcK
  kormedmcqa                           KorMedMCQA
  kbl                                  KBL


### Step B-5: Configure the Model Endpoint

The `ModelConfig` specifies which model endpoint EvalHub should target. This points to your deployed vLLM InferenceService.

#### Key Parameters

| Parameter | Description | Example |
|-----------|-------------|---------|
| `url` | OpenAI-compatible endpoint URL | `https://model-predictor.ns.svc:8443/v1` |
| `name` | Model name (as registered in vLLM) | `vllm-gemma4-e2b` |
| `auth.secret_ref` | K8s Secret for model auth (optional) | `lmeval-sa-token` |

In [22]:
from evalhub import ModelConfig

model = ModelConfig(
    url=BASE_URL,
    name=MODEL_NAME,
)

print("Model Configuration:")
print(f"  URL:   {model.url}")
print(f"  Name:  {model.name}")
print(f"  Auth:  {model.auth or 'None (using cluster-internal access)'}")

Model Configuration:
  URL:   https://vllm-gemma4-e2b-predictor.hyo-project.svc.cluster.local:8443/v1/completions
  Name:  gemma4-e2b-deployment
  Auth:  None (using cluster-internal access)


#### (Optional) Model Authentication

If your InferenceService has OAuth enabled (`security.opendatahub.io/enable-auth: "true"`), reference a Kubernetes Secret containing the ServiceAccount token:

In [23]:
from evalhub.models.api import ModelAuth

model_with_auth = ModelConfig(
    url=BASE_URL,
    name=MODEL_NAME,
    auth=ModelAuth(secret_ref="lmeval-sa-token"),
)

print("Model Configuration (with auth):")
print(f"  URL:        {model_with_auth.url}")
print(f"  Name:       {model_with_auth.name}")
print(f"  Auth:       secret_ref={model_with_auth.auth.secret_ref}")

Model Configuration (with auth):
  URL:        https://vllm-gemma4-e2b-predictor.hyo-project.svc.cluster.local:8443/v1/completions
  Name:       gemma4-e2b-deployment
  Auth:       secret_ref=lmeval-sa-token


### Step B-6: Configure MLflow Experiment Tracking

EvalHub integrates with MLflow to automatically track evaluation metrics, parameters, and artifacts. When you include an `ExperimentConfig` in your job submission, EvalHub will:

1. Create (or reuse) an MLflow experiment with the given name
2. Log all benchmark metrics (accuracy, f1, etc.) as MLflow metrics
3. Tag the run with model info, benchmark details, and custom tags
4. Store detailed result artifacts

The MLflow connection was configured in **Step A-4** via `MLFLOW_TRACKING_URI` in the EvalHub CR.

#### ExperimentConfig in Job Submission

You control experiment tracking per-job via the `experiment` field:

In [24]:
from evalhub import ExperimentConfig, ExperimentTag

experiment = ExperimentConfig(
    name="korean-llm-evaluation",
    tags=[
        ExperimentTag(key="model_family", value="gemma-4"),
        ExperimentTag(key="language", value="korean"),
        ExperimentTag(key="environment", value="dev"),
        ExperimentTag(key="team", value="ai-evaluation"),
    ],
)

print("MLflow Experiment Configuration:")
print(f"  Name:  {experiment.name}")
print(f"  Tags:")
for tag in experiment.tags:
    print(f"    {tag.key}: {tag.value}")

MLflow Experiment Configuration:
  Name:  korean-llm-evaluation
  Tags:
    model_family: gemma-4
    language: korean
    environment: dev
    team: ai-evaluation


### Step B-7: Submit a Single Benchmark Evaluation

Let's submit a simple evaluation using the `korean_lm_eval` provider registered above.

In [25]:
from evalhub import BenchmarkConfig, JobSubmissionRequest

single_job_request = JobSubmissionRequest(
    name="kmmlu-law-evaluation",
    description="Korean MMLU Law benchmark via korean_lm_eval provider",
    tags=["korean", "kmmlu", "law"],
    model=model,
    benchmarks=[
        BenchmarkConfig(
            id="kmmlu_direct_law",
            provider_id=KOREAN_PROVIDER_ID,
            parameters={
                "num_fewshot": 0,
                "limit": 5,
            },
        ),
    ],
    experiment=experiment,
)

print("Job Submission Request:")
print(f"  Name:       {single_job_request.name}")
print(f"  Model:      {single_job_request.model.name} @ {single_job_request.model.url}")
print(f"  Provider:   {KOREAN_PROVIDER_ID}")
print(f"  Benchmarks: {[b.id for b in single_job_request.benchmarks]}")
print(f"  Experiment: {single_job_request.experiment.name}")

Job Submission Request:
  Name:       kmmlu-law-evaluation
  Model:      gemma4-e2b-deployment @ https://vllm-gemma4-e2b-predictor.hyo-project.svc.cluster.local:8443/v1/completions
  Provider:   d949e6fc-b9a2-4402-a920-b118417ec3f5
  Benchmarks: ['kmmlu_direct_law']
  Experiment: korean-llm-evaluation


In [26]:
try:
    job = client.jobs.submit(single_job_request)
    print(f"Job submitted!")
    print(f"  Job ID:        {job.id}")
    print(f"  State:         {job.state}")
    print(f"  MLflow Exp ID: {job.resource.mlflow_experiment_id or 'pending'}")
except Exception as e:
    job = None
    print(f"Failed to submit job: {e}")

Job submitted!
  Job ID:        56bb83c8-a891-45e1-9525-9e1d30cf7c06
  State:         JobStatus.PENDING
  MLflow Exp ID: 1


### Step B-8: Monitor Job Progress

Poll the job status until it completes.

In [27]:
import time
from evalhub import JobStatus

if job is None:
    print("Skipped — no job was submitted in the previous cell.")
else:
    TERMINAL_STATES = {JobStatus.COMPLETED, JobStatus.FAILED, JobStatus.CANCELLED, JobStatus.PARTIALLY_FAILED}

    print(f"Monitoring job {job.id}...")
    print("-" * 60)

    while True:
        status = client.jobs.get(job.id)
        state = status.effective_state

        msg = ""
        if status.status and status.status.message:
            msg = f" - {status.status.message.message}"
        print(f"  [{state.value:>10s}]{msg}")

        if state in TERMINAL_STATES:
            break

        time.sleep(10)

    print("-" * 60)
    print(f"Final state: {state.value}")

Monitoring job 56bb83c8-a891-45e1-9525-9e1d30cf7c06...
------------------------------------------------------------
  [   pending] - Evaluation job created
  [    failed] - Evaluation job is failed. 
Benchmark kmmlu_direct_law failed with message: Evaluation failed: OSError

------------------------------------------------------------
Final state: failed


### Step B-9: View Results

Retrieve the evaluation results, including MLflow run information.

In [28]:
if job is None:
    print("Skipped — no job was submitted.")
else:
    completed_job = client.jobs.get(job.id)

    if completed_job.results:
        print("Evaluation Results:")
        print("=" * 60)

        if completed_job.results.mlflow_experiment_url:
            print(f"\n  MLflow Experiment: {completed_job.results.mlflow_experiment_url}")

        for bm_result in completed_job.results.benchmarks:
            print(f"\n  Benchmark: {bm_result.id} (provider: {bm_result.provider_id})")
            if bm_result.mlflow_run_id:
                print(f"  MLflow Run ID: {bm_result.mlflow_run_id}")
            print(f"  Metrics:")
            for metric_name, metric_value in bm_result.metrics.items():
                print(f"    {metric_name}: {metric_value}")
    else:
        print("No results available yet.")

Evaluation Results:

  MLflow Experiment: http://mlflow.demo.svc.cluster.local:5000/api/2.0/mlflow/experiments

  Benchmark: kmmlu_direct_law (provider: d949e6fc-b9a2-4402-a920-b118417ec3f5)
  Metrics:


### Step B-10: Multi-Benchmark Evaluation

Submit multiple benchmarks in a single request. EvalHub runs them concurrently and tracks all results under one MLflow experiment.

In [29]:
multi_job_request = JobSubmissionRequest(
    name="korean-comprehensive-eval",
    description="Multi-benchmark Korean LLM evaluation",
    tags=["korean", "comprehensive"],
    model=model,
    benchmarks=[
        BenchmarkConfig(
            id="kmmlu_direct_law",
            provider_id=KOREAN_PROVIDER_ID,
            parameters={"num_fewshot": 0, "limit": 5},
        ),
        BenchmarkConfig(
            id="kobest_wic",
            provider_id=KOREAN_PROVIDER_ID,
            parameters={"num_fewshot": 0, "limit": 5},
        ),
        BenchmarkConfig(
            id="arc_easy",
            provider_id="lm_evaluation_harness",
            parameters={"num_fewshot": 0, "limit": 5},
        ),
    ],
    experiment=ExperimentConfig(
        name="korean-comprehensive-evaluation",
        tags=[
            ExperimentTag(key="evaluation_type", value="comprehensive"),
            ExperimentTag(key="model_family", value="gemma-4"),
            ExperimentTag(key="language", value="korean,english"),
        ],
    ),
)

print("Multi-Benchmark Job Request:")
print(f"  Name:       {multi_job_request.name}")
print(f"  Benchmarks: {[b.id for b in multi_job_request.benchmarks]}")
print(f"  Experiment: {multi_job_request.experiment.name}")

# Uncomment to submit:
# multi_job = client.jobs.submit(multi_job_request)
# print(f"\nJob submitted: {multi_job.id}")

Multi-Benchmark Job Request:
  Name:       korean-comprehensive-eval
  Benchmarks: ['kmmlu_direct_law', 'kobest_wic', 'arc_easy']
  Experiment: korean-comprehensive-evaluation


### Step B-11: Use Collections for Standardized Evaluations

Collections group benchmarks into reusable evaluation suites. This is useful for certification or compliance workflows.

In [30]:
try:
    collections = client.collections.list()
    print(f"Available Collections ({len(collections)}):")
    print("=" * 60)
    for coll in collections:
        print(f"\n  Collection: {coll.name}")
        print(f"  ID:         {coll.resource.id}")
        print(f"  Category:   {coll.category}")
        print(f"  Benchmarks: {len(coll.benchmarks)}")
        for bm_ref in coll.benchmarks[:5]:
            print(f"    - {bm_ref.id} (provider: {bm_ref.provider_id})")
        if len(coll.benchmarks) > 5:
            print(f"    ... and {len(coll.benchmarks) - 5} more")
except Exception as e:
    print(f"Failed to list collections: {e}")

Available Collections (3):

  Collection: Toxicity and Ethical Principles
  ID:         toxicity-and-ethical-principles
  Category:   safety
  Benchmarks: 3
    - toxigen (provider: lm_evaluation_harness)
    - truthfulqa_mc1 (provider: lm_evaluation_harness)
    - bigbench_hhh_alignment_multiple_choice (provider: lm_evaluation_harness)

  Collection: Safety & Fairness
  ID:         safety-and-fairness-v1
  Category:   safety
  Benchmarks: 6
    - truthfulqa_mc1 (provider: lm_evaluation_harness)
    - toxigen (provider: lm_evaluation_harness)
    - winogender (provider: lm_evaluation_harness)
    - crows_pairs_english (provider: lm_evaluation_harness)
    - bbq (provider: lm_evaluation_harness)
    ... and 1 more

  Collection: Open LLM Leaderboard v2
  ID:         leaderboard-v2
  Category:   general
  Benchmarks: 6
    - leaderboard_ifeval (provider: lm_evaluation_harness)
    - leaderboard_bbh (provider: lm_evaluation_harness)
    - leaderboard_gpqa (provider: lm_evaluation_harness)

### Step B-12: List and Manage Jobs

Review all submitted evaluation jobs.

In [32]:
try:
    jobs_list = client.jobs.list()
    print(f"Evaluation Jobs ({len(jobs_list)}):")
    print("=" * 60)
    for j in jobs_list:
        state = j.effective_state.value
        exp_name = j.experiment.name if j.experiment else "N/A"
        bm_ids = [b.id for b in j.benchmarks] if j.benchmarks else []
        print(f"  [{state:>16s}] {j.id[:12]}... | {j.name} | exp={exp_name} | benchmarks={bm_ids}")
except Exception as e:
    print(f"Failed to list jobs: {e}")

Evaluation Jobs (1):
  [          failed] 56bb83c8-a89... | kmmlu-law-evaluation | exp=korean-llm-evaluation | benchmarks=['kmmlu_direct_law']


## Reference: EvalHub SDK Quick Reference

### Client SDK Imports

```python
from evalhub import (
    SyncEvalHubClient,          # Synchronous client (recommended for notebooks)
    AsyncEvalHubClient,         # Async client (for production apps)
    ModelConfig,                # Model endpoint configuration
    BenchmarkConfig,            # Benchmark selection and parameters
    JobSubmissionRequest,       # Full job request
    ExperimentConfig,           # MLflow experiment settings
    ExperimentTag,              # MLflow tags
    CollectionRef,              # Reference to a benchmark collection
    EvaluationExports,          # OCI artifact export config
    EvaluationExportsOCI,       # OCI-specific export settings
    OCICoordinates,             # OCI registry coordinates
    JobStatus,                  # Job status enum
)
```

### Key API Patterns

```python
# Initialize client
client = SyncEvalHubClient(
    base_url="https://evalhub:8443",
    auth_token="...",           # Optional: SA token or API key
    insecure=True,              # Skip TLS verification (dev only)
    tenant="my-namespace",      # Kubernetes namespace
)

# Explore resources (all return plain lists)
providers: list[Provider]     = client.providers.list()
benchmarks: list[Benchmark]   = client.benchmarks.list()
collections: list[Collection] = client.collections.list()

# Submit a job
job: EvaluationJob = client.jobs.submit(request)

# Monitor and retrieve results
status: EvaluationJob = client.jobs.get(job.id)
```

### MLflow Experiment Structure

When an `ExperimentConfig` is provided:

- **Experiment Name**: `{prefix}_{experiment.name}`
- **Tags**: Direct mapping from `experiment.tags`
- **Run**: One MLflow run per evaluation request
- **Metrics**: Benchmark scores logged automatically
- **Parameters**: Model config and benchmark settings logged
- **Artifacts**: Detailed result files stored

### Useful Links

- [EvalHub GitHub](https://github.com/eval-hub/eval-hub)
- [EvalHub SDK GitHub](https://github.com/eval-hub/eval-hub-sdk)
- [EvalHub API Docs](https://eval-hub.github.io/eval-hub/)
- [MLflow Integration Guide](https://github.com/eval-hub/eval-hub/blob/main/MLFLOW.md)

## Done!

You've now configured the EvalHub SDK and learned how to:

1. **Connect** to the EvalHub service with the Python SDK
2. **Configure a model endpoint** pointing to your deployed InferenceService
3. **Set up MLflow experiment tracking** with tags and experiment names
4. **Submit evaluations** using lm-evaluation-harness benchmarks
5. **Monitor** job progress and **retrieve results**
6. **Run multi-benchmark** evaluations in a single request

### Next Steps

- **1_builtin_tasks/** -- Run quick evaluations using LMEvalJob (Kubernetes CR)
- **2_custom_tasks/** -- Create custom evaluation tasks with Git-sourced datasets
- **4_eval_hub_benchmark/** -- Analyze benchmark results tracked in EvalHub + MLflow